# The 2026 NeuroGolf Championship

## Score: 5015.20

In [ ]:
from pathlib import Path
import hashlib
import zipfile
import re
import json
import os
import tempfile
import io
import contextlib
from collections import defaultdict, Counter

import numpy as np
import onnx

try:
    import onnxruntime as ort
except Exception:
    ort = None

try:
    import neurogolf_utils as nu
except Exception:
    nu = None

EXPECTED_FILE_COUNT = 401
EXPECTED_PUBLIC_SCORE = "6335.19"
EXPECTED_MANIFEST_SHA256 = "65e5d817b3195de894b8e1bb97045ed8ae6eeba3b4d79ab8dfcb4c4b88aaa13a"
OUTPUT_ZIP = Path("submission.zip")
MAX_ONNX_BYTES = int(1.44 * 1024 * 1024)
BANNED_OPS = {"LOOP", "SCAN", "NONZERO", "UNIQUE", "SCRIPT", "FUNCTION", "COMPRESS"}
MAX_VALID_CANDIDATES_PER_TASK = 512
MAX_PROBE_CANDIDATES = 24
BASE_REPLACE_MARGIN = 8000
FRAGILE_REPLACE_MARGIN = 30000
MIN_PROBE_SCORE = 0.65
FRAGILE_TASKS = {"task101.onnx", "task322.onnx", "task363.onnx"}

input_root = Path("/kaggle/input")
preferred = input_root / "neurogolf-6335-19-controlled-public-artifact"
_seen_roots = set()
search_roots = []
def _add_search_root(p):
    p = Path(p)
    if not p.exists():
        return
    k = p.resolve()
    if k not in _seen_roots:
        _seen_roots.add(k)
        search_roots.append(p)
_add_search_root(preferred)
for _pat in ("*6335*artifact*", "*neurogolf*artifact*", "*blend*", "*neurogolf*"):
    for _p in sorted(input_root.glob(_pat)):
        _add_search_root(_p)
for _z in sorted(input_root.rglob("submission.zip")):
    _add_search_root(_z.parent)
for _hint in input_root.rglob("task001.onnx"):
    _add_search_root(_hint.parent)
use_full_validation = True


def normalize_task_name(name: str):
    m = re.search(r"task(\d{3})", name.lower())
    if not m:
        return None
    return f"task{m.group(1)}.onnx"


def source_rank(source_name: str):
    s = source_name.lower()
    if "6335" in s:
        return 0
    if "controlled" in s:
        return 1
    if "6285" in s or "6233" in s or "6254" in s or "konbu" in s:
        return 2
    if "blend" in s or "beicicc" in s or "afr1ste" in s:
        return 3
    return 9


def manifest_from_items(items):
    lines = []
    for name, data in sorted(items, key=lambda x: x[0]):
        lines.append(f"{name}\t{len(data)}\t{hashlib.sha256(data).hexdigest()}")
    return hashlib.sha256("\n".join(lines).encode()).hexdigest()


def load_zip_models(zip_path: Path):
    out = {}
    with zipfile.ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            if not name.lower().endswith(".onnx"):
                continue
            tn = normalize_task_name(Path(name).name)
            if tn is None:
                continue
            out[tn] = zf.read(name)
    return out


def load_dir_models(root: Path):
    out = {}
    for p in root.rglob("task*.onnx"):
        tn = normalize_task_name(p.name)
        if tn is None:
            continue
        out[tn] = p.read_bytes()
    return out


def one_hot_grid(grid):
    t = np.zeros((1, 10, 30, 30), dtype=np.float32)
    for r, row in enumerate(grid):
        if r >= 30:
            break
        for c, v in enumerate(row):
            if c >= 30:
                break
            if 0 <= v < 10:
                t[0, v, r, c] = 1.0
    return t


def is_processable_model(raw: bytes):
    try:
        if len(raw) > MAX_ONNX_BYTES:
            return False
        model = onnx.load_model_from_string(raw)
        onnx.checker.check_model(model, full_check=True)
        for node in model.graph.node:
            if node.op_type.upper() in BANNED_OPS:
                return False
        inferred = onnx.shape_inference.infer_shapes(model, strict_mode=True)
        graph = inferred.graph
        for item in list(graph.input) + list(graph.value_info) + list(graph.output):
            if not item.type.HasField("tensor_type"):
                continue
            tt = item.type.tensor_type
            if not tt.HasField("shape"):
                return False
            for dim in tt.shape.dim:
                if dim.HasField("dim_param"):
                    return False
                if not dim.HasField("dim_value"):
                    return False
        return True
    except Exception:
        return False


_session_cache = {}

def get_session(raw: bytes):
    if ort is None:
        return True
    key = hashlib.sha256(raw).hexdigest()
    if key in _session_cache:
        return _session_cache[key]
    try:
        sess = ort.InferenceSession(raw, providers=["CPUExecutionProvider"])
    except Exception:
        sess = None
    _session_cache[key] = sess
    return sess

def run_binary_output(raw: bytes, x: np.ndarray):
    if ort is None:
        return None
    sess = get_session(raw)
    if sess is None:
        return None
    try:
        y = sess.run(["output"], {"input": x})[0]
    except Exception:
        return None
    return (y > 0.0).astype(np.float32)

def validate_task_model(raw: bytes, task_payload: dict):
    if ort is None:
        return True
    if get_session(raw) is None:
        return False
    subsets = ["train", "test", "arc-gen"] if use_full_validation else ["train", "test"]
    for subset in subsets:
        for ex in task_payload.get(subset, []):
            x = one_hot_grid(ex["input"])
            pred = run_binary_output(raw, x)
            if pred is None:
                return False
            tgt = one_hot_grid(ex["output"])
            if not np.array_equal(pred, tgt):
                return False
    return True


def estimate_cost(raw: bytes):
    if nu is not None:
        p = None
        try:
            with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as tf:
                tf.write(raw)
                p = tf.name
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                macs, mem, params = nu.score_network(p)
            if None not in (macs, mem, params):
                return int(macs + mem + params)
        except Exception:
            pass
        finally:
            if p is not None:
                try:
                    os.remove(p)
                except Exception:
                    pass
    return len(raw)


def source_candidates(roots):
    zip_sources = []
    dir_sources = []
    for root in roots:
        if not root.exists():
            continue
        zips = sorted(root.rglob("submission.zip"))
        for z in zips:
            zip_sources.append(z)
        if not zips:
            dir_sources.append(root)
    return zip_sources, dir_sources

def build_probe_inputs(task_payload: dict):
    probes = []
    examples = task_payload.get("train", []) + task_payload.get("test", [])
    sizes = []
    colors = set()
    for ex in examples:
        g = ex["input"]
        h = len(g)
        w = len(g[0]) if h else 0
        if h > 0 and w > 0:
            sizes.append((h, w))
        for row in g:
            for v in row:
                if 0 <= v < 10:
                    colors.add(v)
    if not sizes:
        sizes = [(3, 3), (5, 5)]
    sizes = sorted(set(sizes))[:3]
    colors = sorted(colors)[:4]
    if not colors:
        colors = [0, 1]
    for h, w in sizes:
        g1 = [[colors[0] for _ in range(w)] for _ in range(h)]
        probes.append(one_hot_grid(g1))
        c2 = colors[1] if len(colors) > 1 else colors[0]
        g2 = [[(colors[0] if (r + c) % 2 == 0 else c2) for c in range(w)] for r in range(h)]
        probes.append(one_hot_grid(g2))
    return probes[:6]

def probe_consensus_score(task_name: str, task_payload: dict, candidate_pool):
    if ort is None:
        return {src: 1.0 for src, _ in candidate_pool}
    probes = build_probe_inputs(task_payload)
    if not probes:
        return {src: 1.0 for src, _ in candidate_pool}
    pool = candidate_pool[:MAX_PROBE_CANDIDATES]
    trusted = [(src, raw) for src, raw in pool if source_rank(src) <= 3]
    if len(trusted) < 2:
        trusted = pool
    trusted_outputs = []
    for src, raw in trusted:
        outs = []
        ok = True
        for x in probes:
            y = run_binary_output(raw, x)
            if y is None:
                ok = False
                break
            outs.append(hashlib.sha256(y.tobytes()).hexdigest())
        if ok:
            trusted_outputs.append((src, outs))
    if len(trusted_outputs) < 2:
        return {src: 1.0 for src, _ in pool}
    consensus = []
    for i in range(len(probes)):
        cnt = Counter(v[i] for _, v in trusted_outputs)
        consensus.append(cnt.most_common(1)[0][0])
    scores = {}
    for src, raw in pool:
        hits = 0
        seen = 0
        for i, x in enumerate(probes):
            y = run_binary_output(raw, x)
            if y is None:
                continue
            seen += 1
            if hashlib.sha256(y.tobytes()).hexdigest() == consensus[i]:
                hits += 1
        scores[src] = (hits / seen) if seen else 0.0
    return scores


zip_sources, dir_sources = source_candidates(search_roots)

bank = defaultdict(list)
for z in zip_sources:
    try:
        models = load_zip_models(z)
        for k, v in models.items():
            bank[k].append((f"zip:{z.name}", v))
    except Exception:
        pass

for d in dir_sources:
    try:
        models = load_dir_models(d)
        for k, v in models.items():
            bank[k].append((f"dir:{d.name}", v))
    except Exception:
        pass

task_jsons = {}
comp_dir = Path("/kaggle/input/competitions/neurogolf-2026")
if comp_dir.exists():
    for jp in comp_dir.glob("task*.json"):
        tn = normalize_task_name(jp.stem)
        if tn is not None:
            task_jsons[tn] = json.loads(jp.read_text())

source_counts = Counter()
for task_name, items in bank.items():
    for src, _ in items:
        source_counts[src] += 1

anchor_source = None
anchor_candidates = sorted(source_counts.items(), key=lambda kv: (0 if kv[1] >= EXPECTED_FILE_COUNT else 1, source_rank(kv[0]), -kv[1], kv[0]))
if anchor_candidates:
    anchor_source = anchor_candidates[0][0]

selected = {}
usage = Counter()
validation_failures = []
replaced = []
for task_name in sorted(bank.keys()):
    candidates = bank[task_name]
    candidates = sorted(candidates, key=lambda x: (source_rank(x[0]), len(x[1])))
    valid_pool = []
    for src, raw in candidates:
        if not is_processable_model(raw):
            continue
        valid_pool.append((src, raw))
        if len(valid_pool) >= MAX_VALID_CANDIDATES_PER_TASK:
            break

    anchor = None
    if anchor_source is not None:
        for src, raw in valid_pool:
            if src == anchor_source:
                anchor = (src, raw)
                break

    payload = task_jsons.get(task_name)
    probe_scores = {}
    if payload is not None and valid_pool:
        probe_scores = probe_consensus_score(task_name, payload, valid_pool)

    chosen = None
    if anchor is not None:
        chosen = anchor

    if payload is not None:
        if chosen is not None and not validate_task_model(chosen[1], payload):
            chosen = None
        anchor_cost = estimate_cost(chosen[1]) if chosen is not None else None

        best_replacement = None
        best_repl_cost = None
        for src, raw in valid_pool:
            if chosen is not None and src == chosen[0] and raw == chosen[1]:
                continue
            if probe_scores and probe_scores.get(src, 0.0) < MIN_PROBE_SCORE:
                continue
            if not validate_task_model(raw, payload):
                continue
            c = estimate_cost(raw)
            if best_repl_cost is None or c < best_repl_cost or (c == best_repl_cost and source_rank(src) < source_rank(best_replacement[0])):
                best_repl_cost = c
                best_replacement = (src, raw)

        if chosen is None:
            if best_replacement is not None:
                chosen = best_replacement
            elif valid_pool:
                chosen = valid_pool[0]
            elif candidates:
                chosen = candidates[0]
                validation_failures.append(task_name)
        else:
            if best_replacement is not None:
                margin = FRAGILE_REPLACE_MARGIN if task_name in FRAGILE_TASKS else BASE_REPLACE_MARGIN
                if anchor_cost is None or (anchor_cost - best_repl_cost) >= margin:
                    chosen = best_replacement
                    replaced.append(task_name)
    else:
        if chosen is None and valid_pool:
            chosen = valid_pool[0]
        if chosen is None and candidates:
            chosen = candidates[0]

    if chosen is not None:
        selected[task_name] = chosen[1]
        usage[chosen[0]] += 1

if selected:
    manifest = manifest_from_items(list(selected.items()))
    if manifest == EXPECTED_MANIFEST_SHA256:
        print("Matched expected 6335.19 manifest")
    else:
        print("Manifest differs from expected 6335.19")
        print(manifest)

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(selected.keys()):
        zf.writestr(name, selected[name])

zip_sha = hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()
print(f"Wrote {OUTPUT_ZIP}")
print(f"ONNX files: {len(selected)}")
print(f"zip sha256: {zip_sha}")
print(f"sources used: {dict(usage)}")
print(f"anchor source: {anchor_source}")
print(f"replaced tasks: {len(replaced)}")
if replaced:
    print(replaced[:30])
if validation_failures:
    print(f"Validation misses: {len(validation_failures)}")
    print(validation_failures[:20])
if len(selected) != EXPECTED_FILE_COUNT:
    print(f"Warning: expected {EXPECTED_FILE_COUNT}, found {len(selected)}")


## Attribution

This release builds on the public NeuroGolf notebook ecosystem and my previous public artifact. Useful public references include:

- `afr1ste/neurogolf-6285-95-public-score-open-solution`
- `artemnazemtsev/neuro-golf-6312`
- `artemnazemtsev/neurogolf-acking-multiple-tasks`
- `artemnazemtsev/neurogolf-blending-is-all-you-need`
- `magmacot/neurogolf-new-blending`
- `jonathanchan/ngc26-constraint-smart-logic-mix-blending`

The point of listing these is practical traceability: public-score artifacts are easier to learn from when the lineage is explicit.
